In [1]:
import jax
import string
import jax.numpy as jnp
import jax.lax as lax

import numpy as np

### Convert string inputs to a one hot encoded Jax array

In [2]:
def string_to_one_hot(inputs: np.ndarray) -> jnp.ndarray:
    char_to_index = {char: i for i, char in enumerate(string.ascii_uppercase)}

    one_hot_inputs = []
    for row in inputs:
        one_hot_list = []
        for char in row:
            if char.upper() in char_to_index:
                one_hot_vector = np.zeros(len(string.ascii_uppercase))
                one_hot_vector[char_to_index[char.upper()]] = 1
                one_hot_list.append(one_hot_vector)
        one_hot_inputs.append(one_hot_list)

    return jnp.array(one_hot_inputs)


### Create and encode dataset

In [3]:
inputs = np.array([
    ["A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z"],
    ["Z","Y","X","W","V","U","T","S","R","Q","P","O","N","M","L","K","J","I","H","G","F","E","D","C","B","A"],
    ["B","D","F","H","J","L","N","P","R","T","V","X","Z","A","C","E","G","I","K","M","O","Q","S","U","W","Y"],
    ["M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A","B","C","D","E","F","G","H","I","J","K","L"],
    ["H","G","F","E","D","C","B","A","L","K","J","I","P","O","N","M","U","T","S","R","Q","X","W","V","Z","Y"]
  ])

expected = np.array([
    ["B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A"],
    ["A","B","C","D","E","F","G","H","I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z"],
    ["C","E","G","I","K","M","O","Q","S","U","W","Y","A","B","D","F","H","J","L","N","P","R","T","V","X","Z"], 
    ["N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A","B","C","D","E","F","G","H","I","J","K","L","M"],
    ["I","J","K","L","M","N","O","P","Q","R","S","T","U","V","W","X","Y","Z","A","B","C","D","E","F","G","H"]
])

one_hot_inputs = string_to_one_hot(inputs)
one_hot_expected = string_to_one_hot(expected)

print(one_hot_inputs[0])

[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0.]
 [0.

### Initialize hidden layer

In [4]:
seed = 1701
key = jax.random.key(seed)
key, key_W, key_hb = jax.random.split(key, 3)

hidden_size = 128
vocab_size = len(string.ascii_uppercase)
sequence_size = len(one_hot_inputs[0])
alpha=0.0001

W = jax.random.uniform(key=key_W, shape=(hidden_size, hidden_size))
h_bias = jax.random.uniform(key=key_hb, shape=(hidden_size))
h_states = jnp.zeros(shape=(sequence_size, hidden_size))
next_delta_activation = jnp.zeros(shape=(hidden_size))
h_delta_bias = jnp.zeros_like(h_bias)
delta_W = jnp.zeros_like(W)

### Initialize Output Layer

In [5]:
key, key_V, key_ob = jax.random.split(key, 3)

V = jax.random.uniform(key=key_V, shape=(vocab_size, hidden_size))
o_bias = jax.random.uniform(key=key_ob, shape=(vocab_size))
o_states = jnp.zeros(shape=(sequence_size, vocab_size))
o_delta_bias = jnp.zeros_like(o_bias)
delta_V = jnp.zeros_like(V)

### Initialize Input Layer

In [6]:
key, key_U = jax.random.split(key, 2)

U = jax.random.uniform(key=key_U, shape=(hidden_size, vocab_size))
delta_U = jnp.zeros_like(U)


### Initialize Model PyTree

In [7]:
model = {
    'input_layer': {
        'inputs': one_hot_inputs[0],
        'U': U,
        'delta_U': delta_U
    },
    'hidden_layer': {
        'W': W,
        'h_bias': h_bias,
        'h_states': h_states,
        'next_delta_activation': next_delta_activation,
        'h_delta_bias': h_delta_bias,
        'delta_W': delta_W
    },
    'output_layer': {
        'V': V,
        'o_bias': o_bias,
        'o_states': o_states,
        'o_delta_bias': o_delta_bias,
        'delta_V': delta_V
    },
    'config': {
        'vocab_size': vocab_size,
        'sequence_size': sequence_size,
        'hidden_size': hidden_size,
        'alpha': alpha,
        'seed': seed
    }
}

In [8]:
print(model['input_layer']['inputs'].shape, model['hidden_layer']['h_states'].shape, model['output_layer']['o_states'].shape)

(26, 26) (26, 128) (26, 26)


### Feed forward

In [9]:

@jax.jit
def feed_forward(model):

      #Feed forward implementation for a single input
      def feed_forward_step(carry, input):

            #We only need to pull the previous hidden state
            h_state, _ = carry

            #Calculated new hidden state
            weighted_input = jnp.dot(model['input_layer']['U'], input)
            weighted_h_state = jnp.dot(model['hidden_layer']['W'], h_state)
            weighted_sum = weighted_input + weighted_h_state + model['hidden_layer']['h_bias']
            activation = jnp.tanh(weighted_sum) #new h_state

            #Calculate output
            output = jnp.dot(model['output_layer']['V'], activation) + model['output_layer']['o_bias']
            prediction = jax.nn.softmax(output)

            #Return new carry
            new_carry = (activation, prediction)
            return new_carry, new_carry

      #Initial hidden state and output state for carry
      carry_init = (model['hidden_layer']['h_states'][0], model['output_layer']['o_states'][0])

      #Use lax.scan() to loop through all inputs
      final_states, all_states = lax.scan(feed_forward_step, carry_init, model['input_layer']['inputs'])


      return all_states




      

### Loss

In [10]:
def loss(predictions, expected):

    @jax.jit
    def loss_t_step(prediction, expected):
        return -jnp.sum(expected * jnp.log(prediction))
    
    return jnp.sum(jax.vmap(loss_t_step, in_axes=(0, 0))(predictions, expected))



In [12]:
def compute_rnn_loss(model, expected):
    
    #Run feedforward pass
    all_states = feed_forward(model)

    #Update model PyTree with final states
    model['hidden_layer']['h_states'], model['output_layer']['o_states'] = all_states

    print(all_states)

    #Compute loss
    loss_ = loss(model['output_layer']['o_states'], expected)

    return loss_, model

compute_rnn_loss(model, one_hot_expected[0])


(Array([[0.9437667 , 0.5094803 , 0.6464278 , ..., 0.7705177 , 0.83897936,
        0.7718865 ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ]], dtype=float32), Array([[1.49960406e-02, 6.48339167e-02, 2.55928258e-04, 5.98121341e-03,
        1.80816408e-02, 6.01285230e-03, 5.11912629e-03, 2.93469518e-01,
        1.22554833e-02, 2.47785840e-02, 6.67376968e-04, 1.77993421e-02,
        3.23482876e-04, 3.81605350e-05, 8.26112628e-02, 2.12271651e-03,
        3.36332135e-02, 3.89064178e-02, 2.27628499e-01, 2.06479132e-02,
        7.61523610e-03, 6.18038662e-02, 2.5218

(Array(147.35692, dtype=float32),
 {'input_layer': {'inputs': Array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
           0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
          [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0

### Compute Gradient and value

In [ ]:
loss_grad_fn = jax.jit(jax.value_and_grad(compute_rnn_loss, has_aux=True, allow_int=True))

loss_and_model, gradients = loss_grad_fn(model, one_hot_expected[0])

loss, model = loss_and_model

print(gradients)


{'config': {'alpha': Array(0., dtype=float32, weak_type=True), 'hidden_size': array((b'',), dtype=[('float0', 'V')]), 'seed': array((b'',), dtype=[('float0', 'V')]), 'sequence_size': array((b'',), dtype=[('float0', 'V')]), 'vocab_size': array((b'',), dtype=[('float0', 'V')])}, 'hidden_layer': {'W': Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32), 'delta_W': Array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], dtype=float32), 'h_bias': Array([-6.88632997e-03, -1.25482455e-01,  1.30415428e-02, -1.13300279e-01,
       -4.78668869e-01,  8.98647904e-02, -9.72423702e-02, -2.99197212e-02,
       -3.41980

In [20]:
print(model['hidden_layer']['h_states'])

[[0.9437667  0.5094803  0.6464278  ... 0.7705177  0.83897936 0.7718865 ]
 [1.         1.         1.         ... 1.         1.         1.        ]
 [1.         1.         1.         ... 1.         1.         1.        ]
 ...
 [1.         1.         1.         ... 1.         1.         1.        ]
 [1.         1.         1.         ... 1.         1.         1.        ]
 [1.         1.         1.         ... 1.         1.         1.        ]]


### lax.scan test

In [16]:
def cum_sum(carry, slices):
    slice1, slice2 = slices
    carry1, carry2 = carry
    newslice1 = slice1*carry1
    newslice2 = slice2*carry2

    newslices = (newslice1, newslice2)

    return newslices, newslices

finalslices, newarrs = lax.scan(cum_sum, (1,2), (jnp.ones(5), jnp.ones(5)*2))

print(newarrs)

(Array([1., 1., 1., 1., 1.], dtype=float32), Array([ 4.,  8., 16., 32., 64.], dtype=float32))
